# TensorFlow: Fully Convolutional Neural Network for Semantic Segmentation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from common.metric import get_confusion_matrix, get_iou, get_dice

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import Cropping2D
from tensorflow.keras.layers import Add
from tensorflow.keras.layers import Activation

print('TF Version: ', tf.__version__)
print('TF Eager mode: ', tf.executing_eagerly())
print('TF GPU is", "available' if tf.config.list_physical_devices('GPU') else 'not available')

## General

In [ ]:
# The size of batch
BATCH_SIZE = 32
# THe number of epochs
EPOCHS = 50
# Define the list of all classes
CLASS_NAMES = ['sky',
               'building',
               'column/pole',
               'road',
               'side walk',
               'vegetation',
               'traffic light',
               'fence',
               'vehicle',
               'pedestrian',
               'byciclist',
               'void']

## Download Dataset

In [ ]:
# Download dataset (re-sampled CamVid dataset)
url = 'https://storage.googleapis.com/learning-datasets/fcnn-dataset.zip'

data_dir = tf.keras.utils.get_file(
    origin=url,
    extract=True)

In [ ]:
# Define images and labels (classification maps) paths of training and validation datasets
TR_IMAGES_PATH = os.path.join(data_dir, 'dataset1', 'images_prepped_train')
TR_LABELS_PATH = os.path.join(data_dir, 'dataset1', 'annotations_prepped_train')
VL_IMAGES_PATH = os.path.join(data_dir, 'dataset1', 'images_prepped_test')
VL_LABELS_PATH = os.path.join(data_dir, 'dataset1', 'annotations_prepped_test')

In [ ]:
TR_NUM_IMAGES = len(os.listdir(TR_IMAGES_PATH))
VL_NUM_IMAGES = len(os.listdir(VL_IMAGES_PATH))
print(f'Number of training images: {TR_NUM_IMAGES}')
print(f'Number of validation images: {VL_NUM_IMAGES}')

## Prepare Dataset

In [ ]:
def map_filename_to_image_and_mask(t_filename, a_filename, height=224, width=224):
    '''
        Preprocesses the dataset by:
        * resizing the input image and label maps
        * normalizing the input image pixels
        * reshaping the label maps from (height, width, 1) to (height, width, 12)
    :param t_filename: path to the raw input image
    :param a_filename: path to the raw annotation (label map) file
    :param height: height in pixels to resize to
    :param width: width in pixels to resize to
    :return:
    '''
    # Convert image and mask files to tensors
    img_raw = tf.io.read_file(t_filename)
    anno_raw = tf.io.read_file(a_filename)
    image = tf.image.decode_jpeg(img_raw)
    annotation = tf.image.decode_jpeg(anno_raw)

    # Resize image and segmentation mask
    image = tf.image.resize(image, (height, width,))
    annotation = tf.image.resize(annotation, (height, width,))
    image = tf.reshape(image, (height, width, 3,))
    annotation = tf.cast(annotation, dtype=tf.int32)
    annotation = tf.reshape(annotation, (height, width, 1,))
    stack_list = []

    # Reshape segmentation masks
    for c in range(len(CLASS_NAMES)):
        mask = tf.equal(annotation[:,:,0], tf.constant(c))
        stack_list.append(tf.cast(mask, dtype=tf.int32))
    annotation = tf.stack(stack_list, axis=2)

    # Normalize pixels in the input image
    image = image/127.5
    image -= 1

    return image, annotation

In [ ]:
def get_dataset_slice_paths(image_dir, label_map_dir):
    '''
    Generates the lists of image and label map paths
    :param image_dir: path to the input images directory
    :param label_map_dir: path to the label map directory
    :return:
        image_paths (list of strings) - paths to each image file
        label_map_paths (list of strings) - paths to each label map
    '''
    image_file_list = os.listdir(image_dir)
    label_map_file_list = os.listdir(label_map_dir)
    image_paths = [os.path.join(image_dir, fname) for fname in image_file_list]
    label_map_paths = [os.path.join(label_map_dir, fname) for fname in label_map_file_list]
    return image_paths, label_map_paths

In [ ]:
def get_training_dataset(image_paths, label_map_paths):
    '''
    Prepares shuffled batches of the training set.
    :param image_paths: paths to each image file in the train set
    :param label_map_paths: paths to each label map in the train set
    :return: dataset containing the preprocessed train set
    '''
    training_dataset = tf.data.Dataset.from_tensor_slices((image_paths, label_map_paths))
    training_dataset = training_dataset.map(map_filename_to_image_and_mask)
    training_dataset = training_dataset.shuffle(100, reshuffle_each_iteration=True)
    training_dataset = training_dataset.batch(BATCH_SIZE)
    training_dataset = training_dataset.repeat()
    training_dataset = training_dataset.prefetch(-1)

    return training_dataset

In [ ]:
def get_validation_dataset(image_paths, label_map_paths):
    '''
    Prepares batches of the validation set.
    :param image_paths: paths to each image file in the val set
    :param label_map_paths: paths to each label map in the val set
    :return: dataset containing the preprocessed validation set
    '''
    validation_dataset = tf.data.Dataset.from_tensor_slices((image_paths, label_map_paths))
    validation_dataset = validation_dataset.map(map_filename_to_image_and_mask)
    validation_dataset = validation_dataset.batch(BATCH_SIZE)
    validation_dataset = validation_dataset.repeat()
    return validation_dataset

In [ ]:
# Get the full paths to the images
tr_image_paths, tr_label_paths = get_dataset_slice_paths(TR_IMAGES_PATH, TR_LABELS_PATH)
vl_image_paths, vl_label_paths = get_dataset_slice_paths(VL_IMAGES_PATH, VL_LABELS_PATH)

# Generate the train and val datasets
tr_ds = get_training_dataset(tr_image_paths, tr_label_paths)
vl_ds = get_validation_dataset(vl_image_paths, vl_label_paths)

In [ ]:
images, labels = next(iter(tr_ds.take(1)))
img_shape = tf.shape(images).numpy()
lbl_shape = tf.shape(labels).numpy()

print('Dataset Info:')
print(f'...images: batchSize<{img_shape[0]}>, dims<{img_shape[1]}x{img_shape[2]}>, channels<{img_shape[3]}>')
print(f'...labels: batchSize<{lbl_shape[0]}>, dims<{lbl_shape[1]}x{lbl_shape[2]}>, classes<{lbl_shape[3]}>')

## Define Model

In [ ]:
# Download pre-trained weights of VGG16 network
weights_file = tf.keras.utils.get_file(
    origin='https://github.com/'
           'fchollet/deep-learning-models/releases/download/v0.1/'
           'vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5'
)

In [ ]:
def SVGG16(n_classes, weights_path=None):
    inputs = tf.keras.layers.Input(shape=(224,224,3))

    ### Encoder ###

    # Block 1
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(inputs)
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)
    p1 = x
    # Block 2
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)
    p2 = x
    # Block 3
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)
    p3 = x
    # Block 4
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)
    p4 = x
    # Block 5
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv1')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv2')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv3')(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)

    # Load pre-trained weights for previously defined blocks (1-5)
    if weights_path is not None:
        vgg = Model(inputs=inputs, outputs=x)
        vgg.load_weights(weights_path)

    # Add extra layers to extract more high-level features
    x = Conv2D(4096, (7,7), activation='relu', padding='same')(x)
    x = Conv2D(4096, (1,1), activation='relu', padding='same')(x)
    p5 = x

    ### Decoder ###

    # Upsample p5 by 2x and crop
    x = Conv2DTranspose(n_classes, (4,4), strides=(2,2), use_bias=False)(p5)
    x = Cropping2D(cropping=(1,1))(x)
    d1 = x

    # Reshape p4 prediction to a shape of d1
    x = Conv2D(n_classes, (1,1), activation='relu' , padding='same')(p4)
    # Summate p4 with p5
    x = Add()([d1, x])

    # Upsample the sum p4+p5 by 2x
    x = Conv2DTranspose(n_classes, (4,4), strides=(2,2), use_bias=False )(x)
    x = Cropping2D(cropping=(1,1))(x)
    d2 = x

    # Reshape p3 prediction to a shape of d2
    x = Conv2D(n_classes, (1,1), activation='relu' , padding='same')(p3)
    # Summate p3 with the sum p4+p5
    x = Add()([d2, x])

    # Upsample to the size of the original image
    x = Conv2DTranspose(n_classes , kernel_size=(8,8) ,  strides=(8,8) , use_bias=False )(x)
    # Add the class predictions
    output = Activation('softmax')(x)

    return Model(inputs=inputs, outputs=output)

In [ ]:
# Create instance of the network
model = SVGG16(n_classes=12, weights_path=weights_file)

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9, nesterov=True),
              metrics=['accuracy'])

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True)

## Train Model

In [ ]:
train_steps = TR_NUM_IMAGES // BATCH_SIZE
valid_steps = VL_NUM_IMAGES // BATCH_SIZE

history = model.fit(
    tr_ds,
    steps_per_epoch=train_steps,
    validation_data=vl_ds,
    validation_steps=valid_steps,
    epochs=EPOCHS)

## Evaluate Model

In [ ]:
n_classes = len(CLASS_NAMES)

In [ ]:
def get_images_and_labels_test_arrays(count):
    y_true_labels = []
    y_true_images = []
    ds = vl_ds.unbatch()
    ds = ds.batch(VL_NUM_IMAGES)
    for images, labels in ds.take(1):
        y_true_images = images.numpy()
        y_true_labels = labels.numpy()
    y_true_labels = y_true_labels[:count]
    y_true_labels = np.argmax(y_true_labels, axis=3)
    return y_true_images, y_true_labels

In [ ]:
# Get true images and labels
_, y_true_labels = get_images_and_labels_test_arrays(count=96)

In [ ]:
# Get pred labels
y_pred_labels = np.argmax(model.predict(vl_ds, steps=valid_steps), axis=3)

In [ ]:
# Calculate global confusion matrix over all samples
cms = []
gcm = np.full(shape=(n_classes, n_classes), fill_value=0, dtype=np.int64)
for (y_pred, y_true) in zip(y_pred_labels, y_true_labels):
    cm = get_confusion_matrix(y_pred, y_true, n_classes)
    cms.append(cm)
    gcm += cm

In [ ]:
# Display confusion matrix of particular sample
sample = 0
ds = ConfusionMatrixDisplay(
    confusion_matrix=cms[sample],
    display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 8))
ds.plot(colorbar=False, ax=ax)
plt.show()

In [ ]:
# Calculates class-wise IoU metric
IoU = get_iou(gcm)
# Calculates class-wise Dice metric
Dice = get_dice(gcm)

In [ ]:
df = pd.DataFrame(
    np.stack([IoU, Dice], axis=1),
    index=CLASS_NAMES,
    columns=['IoU', 'Dice'])

df